# Knowledge retrieval

Build the index first with `.venv/bin/python scripts/build_knowledge.py --data-dir data/raw`.
Use the project kernel. The saved index contains source passages, not generated incident explanations.

## Source inventory

Check how many documents come from the supplied runbooks, project notes, and incident records.
Later incident records are excluded using the stored cutoff.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from banking_observability.knowledge_retrieval import KnowledgeStore, document_text

store = KnowledgeStore(ROOT / "models/knowledge")
changes = store.source_changes(ROOT / "data/raw")
if changes:
    raise ValueError(f"Sources changed; rebuild the index: {changes}")
print("Knowledge cutoff:", store.manifest["sources"]["cutoff"])
display(pd.Series(store.manifest["sources"]["documents_by_kind"], name="documents"))

## A payment query

Search for the actual NEFT gateway-timeout wording. Read the returned source spans, not only the score.
One passage per document is returned; the rest of a document can be opened separately.

In [ ]:
reply = store.search("NEFT transaction gateway timeout")
display(pd.DataFrame([{k: hit[k] for k in ("document_id", "title", "kind", "score")} for hit in reply["results"]]))
for hit in reply["results"]:
    print(hit["chunk_id"])
    print(hit["text"])
    for ref in hit["references"]:
        print(f"  {ref['path']}:{ref['line_start']}-{ref['line_end']}")
    print()

## Compare the FAISS scores with the vectors

This index uses normalized TF-IDF vectors. Recompute the dot products independently and compare them
with the native search output. This checks the implementation, not whether each passage is useful.

In [ ]:
query_vector = store.encoder.transform([reply["query"]]).toarray().astype("float32")
passage_vectors = store.encoder.transform([document_text(c) for c in store.chunks]).toarray().astype("float32")
scores, indices = store.index.search(query_vector, store.index.ntotal)
expected = passage_vectors @ query_vector[0]
np.testing.assert_allclose(scores[0], expected[indices[0]], atol=1e-6)
print("FAISS scores match the independent dot products.")

## Keep the incident-source disagreement visible

Open an incident card with conflicting KPIs. The references include the incident row and the monitoring
row at the same timestamp. Neither the supplied severity label nor a similar past record proves a root cause.

In [ ]:
incident = store.get_document("incident:INC000012")[0]
display(pd.DataFrame(incident["monitoring_disagreements"]))
for ref in incident["references"]:
    print(f"{ref['path']}:{ref['line_start']}-{ref['line_end']}")
    print(ref["excerpt"])
print(incident["text"].splitlines()[-1])

## Query checks

The example questions use wording from the indexed topics. Also try one query with no shared vocabulary.
This small set does not tell us how search will perform on new questions.

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate_retrieval import evaluate
questions = json.loads((ROOT / "config/retrieval_queries.json").read_text())
result = evaluate(store, questions)
display(pd.DataFrame(result["rows"])[["query", "rank", "passed", "status"]])
print("Known-topic hit rate at 3:", result["hit_rate_at_k"])
print(store.search("telescope orbital galaxy"))